In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/sales_flat_clean.csv", dtype={"postal_code": str})
print(df.shape)
df.info()

(9993, 22)
<class 'pandas.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         9993 non-null   int64  
 1   order_id       9993 non-null   str    
 2   order_date     9993 non-null   str    
 3   ship_date      9993 non-null   str    
 4   ship_mode      9993 non-null   str    
 5   customer_id    9993 non-null   str    
 6   customer_name  9993 non-null   str    
 7   segment        9993 non-null   str    
 8   country        9993 non-null   str    
 9   state          9993 non-null   str    
 10  city           9993 non-null   str    
 11  postal_code    9993 non-null   str    
 12  region         9993 non-null   str    
 13  product_key    9993 non-null   int64  
 14  product_id     9993 non-null   str    
 15  category       9993 non-null   str    
 16  sub_category   9993 non-null   str    
 17  product_name   9993 non-null   str    
 18  sales   

In [2]:
print("Missing values (NaN) in the whole table:", df.isna().sum().sum())

for col in df.columns:
    blanks = (df[col].astype(str).str.strip() == "").sum()
    if blanks > 0:
        print("Blank text in column:", col, blanks)
print("Blank check finished")

Missing values (NaN) in the whole table: 0
Blank check finished


In [3]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Repeated row_id:", df["row_id"].duplicated().sum())

other_cols = [c for c in df.columns if c != "row_id"]
print("Repeated rows when ignoring row_id:", df.duplicated(subset=other_cols).sum())

Fully duplicated rows: 0
Repeated row_id: 0
Repeated rows when ignoring row_id: 0


In [4]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%Y-%m-%d")
df["ship_date"]  = pd.to_datetime(df["ship_date"],  format="%Y-%m-%d")

print(df[["order_date", "ship_date"]].dtypes)

order_date    datetime64[us]
ship_date     datetime64[us]
dtype: object


In [5]:
print("First order:", df["order_date"].min())
print("Last order:", df["order_date"].max())
print("Shipped before ordered:", (df["ship_date"] < df["order_date"]).sum())
print(df["order_date"].dt.year.value_counts().sort_index())

First order: 2014-01-03 00:00:00
Last order: 2017-12-30 00:00:00
Shipped before ordered: 0
order_date
2014    1992
2015    2102
2016    2587
2017    3312
Name: count, dtype: int64


In [6]:
print(df[["sales", "quantity", "discount", "profit"]].describe().round(2))

print("Sales <= 0:", (df["sales"] <= 0).sum())
print("Quantity < 1:", (df["quantity"] < 1).sum())
print("Discount outside 0 to 1:", ((df["discount"] < 0) | (df["discount"] > 1)).sum())
print("Negative profit rows:", (df["profit"] < 0).sum())
print("Zero profit rows:", (df["profit"] == 0).sum())

          sales  quantity  discount   profit
count   9993.00   9993.00   9993.00  9993.00
mean     229.85      3.79      0.16    28.66
std      623.28      2.23      0.21   234.27
min        0.44      1.00      0.00 -6599.98
25%       17.28      2.00      0.00     1.73
50%       54.48      3.00      0.20     8.67
75%      209.94      5.00      0.20    29.36
max    22638.48     14.00      0.80  8399.98
Sales <= 0: 0
Quantity < 1: 0
Discount outside 0 to 1: 0
Negative profit rows: 1870
Zero profit rows: 65


In [7]:
for col in ["ship_mode", "segment", "country", "region", "category"]:
    print(df[col].value_counts())
    print()

print("Different sub-categories:", df["sub_category"].nunique())
print("Different states:", df["state"].nunique())

text_cols = ["order_id", "ship_mode", "customer_id", "customer_name", "segment", "country",
             "state", "city", "postal_code", "region", "product_id", "category",
             "sub_category", "product_name"]
for col in text_cols:
    n = (df[col] != df[col].str.strip()).sum()
    if n > 0:
        print("Extra spaces in:", col, n)
print("Space check finished")

ship_mode
Standard Class    5967
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64

segment
Consumer       5191
Corporate      3020
Home Office    1782
Name: count, dtype: int64

country
United States    9993
Name: count, dtype: int64

region
West       3203
East       2847
Central    2323
South      1620
Name: count, dtype: int64

category
Office Supplies    6026
Furniture          2120
Technology         1847
Name: count, dtype: int64

Different sub-categories: 17
Different states: 49
Space check finished


In [8]:
short = df[df["postal_code"].str.len() == 4]
print("Postal codes with only 4 digits:", len(short))
print(short["state"].value_counts())

df["postal_code"] = df["postal_code"].str.zfill(5)
print(df["postal_code"].str.len().value_counts())

Postal codes with only 4 digits: 438
state
Massachusetts    135
New Jersey       130
Connecticut       82
Rhode Island      56
New Hampshire     27
Maine              8
Name: count, dtype: int64
postal_code
5    9993
Name: count, dtype: int64


In [9]:
for col in ["sales", "profit"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    below = (df[col] < low).sum()
    above = (df[col] > high).sum()
    print(col, "| Q1:", round(q1, 2), "| Q3:", round(q3, 2),
          "| limits:", round(low, 2), "to", round(high, 2),
          "| below:", below, "| above:", above)

cols_to_show = ["order_id", "product_name", "sales", "discount", "profit"]
print("\nBiggest sales lines")
print(df.nlargest(5, "sales")[cols_to_show])
print("\nBiggest loss lines")
print(df.nsmallest(5, "profit")[cols_to_show])

sales | Q1: 17.28 | Q3: 209.94 | limits: -271.71 to 498.93 | below: 0 | above: 1167
profit | Q1: 1.73 | Q3: 29.36 | limits: -39.72 to 70.81 | below: 604 | above: 1277

Biggest sales lines
            order_id                                       product_name  \
8784  CA-2014-145317  Cisco TelePresence System EX90 Videoconferenci...   
9364  CA-2016-118689              Canon imageCLASS 2200 Advanced Copier   
7777  CA-2017-140151              Canon imageCLASS 2200 Advanced Copier   
9213  CA-2017-127180              Canon imageCLASS 2200 Advanced Copier   
4019  CA-2017-166709              Canon imageCLASS 2200 Advanced Copier   

          sales  discount     profit  
8784  22638.480       0.5 -1811.0784  
9364  17499.950       0.0  8399.9760  
7777  13999.960       0.0  6719.9808  
9213  11199.968       0.2  3919.9888  
4019  10499.970       0.0  5039.9856  

Biggest loss lines
            order_id                               product_name     sales  \
2192  CA-2016-108196  Cubify C

In [10]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_month_name"] = df["order_date"].dt.month_name()
df["year_month"] = df["order_date"].dt.strftime("%Y-%m")
df["days_to_ship"] = (df["ship_date"] - df["order_date"]).dt.days

print(df[["order_date", "order_year", "order_month", "order_month_name", "year_month", "days_to_ship"]].head())
print(df["days_to_ship"].value_counts().sort_index())
print(df.shape)

  order_date  order_year  order_month order_month_name year_month  \
0 2014-03-31        2014            3            March    2014-03   
1 2014-03-31        2014            3            March    2014-03   
2 2014-09-15        2014            9        September    2014-09   
3 2014-09-15        2014            9        September    2014-09   
4 2015-10-04        2015           10          October    2015-10   

   days_to_ship  
0             5  
1             5  
2             5  
3             5  
4             3  
days_to_ship
0     519
1     369
2    1334
3    1005
4    2773
5    2169
6    1203
7     621
Name: count, dtype: int64
(9993, 27)


In [11]:
df.to_csv("../data/processed/sales_analysis_ready.csv", index=False)

check = pd.read_csv("../data/processed/sales_analysis_ready.csv", dtype={"postal_code": str})
print(check.shape)
print("Total sales:", round(check["sales"].sum(), 2))
print(check["postal_code"].str.len().value_counts())

(9993, 27)
Total sales: 2296919.49
postal_code
5    9993
Name: count, dtype: int64


## Cleaning summary

| Problem | What we found | Action | Why |
|---|---|---|---|
| Repeated row | Row ID 3407 repeated Row ID 3406 in every other column | Left out in SQL (kept in staging) | Counting one sale twice inflates totals. This is an assumption |
| Hidden characters | 225 product names | Replaced with normal spaces in SQL | They break text matching in Python and Power BI |
| Product IDs with two names | 32 Product IDs | Kept both names using product_key | Different products shared one ID |
| Missing / blank values | 0 missing values and 0 blank texts in 9,993 rows | None needed | Nothing to fix |
| Dates stored as text | order_date and ship_date | Converted to real dates in Python | Needed for time analysis |
| Impossible dates | 0 shipped before ordered | None needed | A ship date before the order date is impossible, and none exist |
| Invalid sales / quantity / discount | Sales <= 0: 0, Quantity < 1: 0, Discount outside 0 to 1: 0 | None needed | All values are valid |
| Negative profit | 1870 lines | Kept | Real losses, not errors |
| Postal codes with 4 digits | 438 rows in 6 states | Added a leading zero | ZIP codes are 5 digits |
| Outliers in sales and profit | 1167 sales, 1881 profit lines flagged | Kept | Real big sales and real losses. Studied in EDA |